# Deploy MCP Servers on OpenShift

This notebook deploys MCP servers as shared services on OpenShift, accessible by all team members via Routes.

**Servers to deploy:**
1. Sequential Thinking — Structured problem solving (**fully local, always deployed**)
2. Context7 — Library documentation (requires internet, optional)
3. GitHub — Repository operations (requires internet, optional)
4. gh-grep — GitHub code search (requires internet, optional)
5. Playwright — Browser automation (requires internet, optional)
6. Code Sandbox — Secure code execution (**fully local, always deployed**)

> Internet-dependent servers auto-skip when connectivity is unavailable.

## 1. Verify Cluster Access

In [5]:
%%bash
echo "Cluster: $(oc whoami --show-server)"
echo "User: $(oc whoami)"
echo ""
echo "Apps domain (for Route URLs):"
oc get ingresses.config cluster -o jsonpath='{.spec.domain}'
echo ""

Cluster: https://api.openshift-cluster.sandbox1785.opentlc.com:6443
User: kube:admin

Apps domain (for Route URLs):
apps.openshift-cluster.sandbox1785.opentlc.com


## 2. Create Namespace and Secrets

All MCP servers deploy into the `mcp-servers` namespace.

In [2]:
%%bash
# Create namespace
oc apply -f manifests/00-namespace-secret.yaml

echo ""
echo "⚠️  IMPORTANT: Update the secret with your actual tokens:"
echo ""
echo "  oc set data secret/mcp-api-keys -n mcp-servers \\"
echo "    --from-literal=GITHUB_TOKEN=ghp_your-actual-token"

namespace/mcp-servers created
secret/mcp-api-keys created

⚠️  IMPORTANT: Update the secret with your actual tokens:

  oc set data secret/mcp-api-keys -n mcp-servers \
    --from-literal=GITHUB_TOKEN=ghp_your-actual-token


In [3]:
import os
from dotenv import load_dotenv
import subprocess

load_dotenv("../.env")

github_token = os.getenv("GITHUB_TOKEN")

if github_token and github_token != "ghp_your-github-token-here":
    subprocess.run([
        "oc", "create", "secret", "generic", "mcp-api-keys",
        f"--from-literal=GITHUB_TOKEN={github_token}",
        "-n", "mcp-servers",
        "--dry-run=client", "-o", "yaml"
    ], capture_output=False)
    # Pipe to oc apply if you want auto-update:
    # | oc apply -f -
    print("✅ Tokens loaded from .env")
else:
    print("⚠️  Set GITHUB_TOKEN in .env, then re-run or use oc command above.")

⚠️  Set GITHUB_TOKEN in .env, then re-run or use oc command above.


## 3. Server 1 — Context7 (Requires Internet)

Context7 provides up-to-date library documentation for AI agents.

The MCP server process can run locally, but it **always calls Upstash's external API** to fetch docs (the crawling/parsing backend is proprietary). This means:

| Environment | Works? | How |
|-------------|--------|-----|
| Internet access ✅ | Yes | Deploy as Pod in cluster or use remote endpoint directly |
| Air-gapped / disconnected ❌ | No | Not supported (Enterprise On-Premise license required) |

Below deploys Context7 as a Pod in the cluster (still needs outbound internet). **Skip this cell if disconnected.**

**Tools:** `resolve-library-id`, `get-library-docs`

In [4]:
%%bash
# Check internet connectivity, then deploy Context7 as a local Pod
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://mcp.context7.com/mcp)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "405" ]; then
    echo "✅ Internet reachable — deploying Context7 as cluster Pod..."
    oc apply -f manifests/05-context7.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-context7 -n mcp-servers --timeout=120s 2>/dev/null \
        && echo "✅ Context7 Pod ready" \
        || echo "⏳ Pod still starting... check: oc get pods -n mcp-servers"
    echo ""
    echo "Route URL:"
    oc get route mcp-context7 -n mcp-servers -o jsonpath='https://{.spec.host}/mcp' 2>/dev/null
    echo ""
else
    echo "⚠️  No internet access (HTTP $HTTP_CODE) — skipping Context7."
    echo "   This server requires outbound connectivity to Upstash API."
    echo "   The rest of the lab works without it."
fi

Testing Context7 connectivity...
✅ Context7 reachable (HTTP 405)
   Endpoint: https://mcp.context7.com/mcp
   No deployment needed.


## 4. Server 2 — Sequential Thinking

Wraps `@modelcontextprotocol/server-sequential-thinking` (stdio) with `supergateway` to expose as Streamable HTTP.

In [ ]:
%%bash
echo "Deploying Sequential Thinking MCP server..."
oc apply -f manifests/01-sequential-thinking.yaml

echo ""
echo "Waiting for pod to be ready..."
oc wait --for=condition=available deployment/mcp-sequential-thinking -n mcp-servers --timeout=120s

echo ""
echo "Route URL:"
oc get route mcp-sequential-thinking -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""

## 5. Server 3 — GitHub MCP (Requires Internet)

Full GitHub API access via `GITHUB_TOKEN`. **Requires outbound internet** to reach `api.github.com`.

| Environment | Works? |
|-------------|--------|
| Internet access ✅ | Yes |
| Air-gapped / disconnected ❌ | No — Skip this cell |

**Skip this cell if disconnected.**

In [ ]:
%%bash
echo "Checking outbound internet access to GitHub API..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://api.github.com)

if [ "$HTTP_CODE" = "200" ]; then
    echo "✅ GitHub API reachable — deploying GitHub MCP server..."
    oc apply -f manifests/02-github.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-github -n mcp-servers --timeout=120s 2>/dev/null \
        && echo "✅ GitHub MCP ready" \
        || echo "⏳ Pod still starting..."
    echo ""
    echo "Route URL:"
    oc get route mcp-github -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "⚠️  No internet access to GitHub API (HTTP $HTTP_CODE) — skipping."
    echo "   This server requires outbound connectivity to api.github.com."
    echo "   The rest of the lab works without it."
fi

## 6. Server 4 — gh-grep (Requires Internet)

Custom MCP server for GitHub code search. **Requires outbound internet** to reach `api.github.com`.

| Environment | Works? |
|-------------|--------|
| Internet access ✅ | Yes |
| Air-gapped / disconnected ❌ | No — Skip this cell |

**Skip this cell if disconnected.**

In [ ]:
%%bash
echo "Checking outbound internet access to GitHub API..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://api.github.com)

if [ "$HTTP_CODE" = "200" ]; then
    echo "✅ GitHub API reachable — deploying gh-grep MCP server..."
    oc apply -f manifests/03-gh-grep.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-gh-grep -n mcp-servers --timeout=120s 2>/dev/null \
        && echo "✅ gh-grep MCP ready" \
        || echo "⏳ Pod still starting..."
    echo ""
    echo "Route URL:"
    oc get route mcp-gh-grep -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "⚠️  No internet access to GitHub API (HTTP $HTTP_CODE) — skipping."
    echo "   This server requires outbound connectivity to api.github.com."
    echo "   The rest of the lab works without it."
fi

## 7. Server 5 — Playwright (Requires Internet)

Browser automation with Microsoft Playwright. Provides `navigate`, `click`, `fill`, `screenshot`, `pdf` tools for AI agents.

| Environment | Works? |
|-------------|--------|
| Internet access ✅ | Yes — can browse any website |
| Air-gapped / disconnected ❌ | Partial — only local/cluster URLs |

**Skip this cell if disconnected and no local web targets are needed.**

In [ ]:
%%bash
echo "Checking outbound internet access..."
HTTP_CODE=$(curl -s -o /dev/null -w "%{http_code}" -m 5 https://www.google.com)

if [ "$HTTP_CODE" = "200" ] || [ "$HTTP_CODE" = "301" ] || [ "$HTTP_CODE" = "302" ]; then
    echo "✅ Internet reachable — deploying Playwright MCP server..."
    # Remove old Chrome DevTools if present
    oc delete deployment mcp-chrome-devtools -n mcp-servers 2>/dev/null && echo "   (removed old chrome-devtools)"
    oc delete svc mcp-chrome-devtools -n mcp-servers 2>/dev/null
    oc delete route mcp-chrome-devtools -n mcp-servers 2>/dev/null

    oc apply -f manifests/04-playwright.yaml
    echo ""
    oc wait --for=condition=available deployment/mcp-playwright -n mcp-servers --timeout=180s 2>/dev/null \
        && echo "✅ Playwright MCP ready" \
        || echo "⏳ Pod still starting (Playwright image is large)..."
    echo ""
    echo "Route URL:"
    oc get route mcp-playwright -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
    echo ""
else
    echo "⚠️  No internet access (HTTP $HTTP_CODE) — skipping Playwright."
    echo "   This server needs internet to browse external websites."
    echo "   The rest of the lab works without it."
fi

## 7. Server 6: Code Sandbox (Local — Always Deployed)

Secure code execution sandbox powered by OpenShell (NVIDIA + Red Hat).
Runs Python, Bash, and Node.js code in an isolated workspace.
**No internet required** — works fully offline.

In [ ]:
%%bash
echo "Deploying Code Sandbox MCP server (local, no internet needed)..."
oc apply -f manifests/06-code-sandbox.yaml

echo ""
oc wait --for=condition=available deployment/mcp-code-sandbox -n mcp-servers --timeout=180s 2>/dev/null \
    && echo "✅ Code Sandbox MCP ready" \
    || echo "⏳ Pod still starting..."

echo ""
echo "Route URL:"
oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='https://{.spec.host}/mcp'
echo ""
echo ""
echo "Health check:"
ROUTE=$(oc get route mcp-code-sandbox -n mcp-servers -o jsonpath='{.spec.host}')
curl -sk "https://${ROUTE}/health"
echo ""

## 8. Verify All Servers

In [ ]:
%%bash
echo "MCP Server Deployment Status"
echo "============================================================"
echo ""
echo "=== Pods ==="
oc get pods -n mcp-servers -o wide

echo ""
echo "=== Routes (IDE Endpoints) ==="
echo ""
printf "%-25s %s\n" "SERVER" "ENDPOINT"
printf "%-25s %s\n" "-------" "--------"

for route in $(oc get routes -n mcp-servers -o jsonpath='{.items[*].metadata.name}'); do
    host=$(oc get route $route -n mcp-servers -o jsonpath='{.spec.host}')
    printf "%-25s %s\n" "$route" "https://${host}/mcp"
done

In [ ]:
import subprocess

result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

print("Health Check (deployed servers):")
print("=" * 60)

for line in result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        url = f"https://{host}/mcp"
        r = subprocess.run(["curl", "-sk", "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
                          capture_output=True, text=True)
        status = "✅" if r.stdout.strip() in ["200", "405"] else "❌"
        print(f"{status} {name}: {url}")

if not result.stdout.strip():
    print("⚠️  No routes found. Deploy servers first.")

## Summary

| Server | Deployment | Internet Required | Route |
|--------|-----------|:-----------------:|-------|
| Sequential Thinking | OpenShift Pod | ❌ No | `https://mcp-sequential-thinking-mcp-servers.apps.CLUSTER/mcp` |
| Context7 | OpenShift Pod | ✅ Yes (optional) | `https://mcp-context7-mcp-servers.apps.CLUSTER/mcp` |
| GitHub | OpenShift Pod | ✅ Yes (optional) | `https://mcp-github-mcp-servers.apps.CLUSTER/mcp` |
| gh-grep | OpenShift Pod | ✅ Yes (optional) | `https://mcp-gh-grep-mcp-servers.apps.CLUSTER/mcp` |
| Playwright | OpenShift Pod | ✅ Yes (optional) | `https://mcp-playwright-mcp-servers.apps.CLUSTER/mcp` |
| Code Sandbox | OpenShift Pod | ❌ No | `https://mcp-code-sandbox-mcp-servers.apps.CLUSTER/mcp` |

> **Disconnected 환경에서는**: Sequential Thinking + Code Sandbox는 필수 배포.
> Context7, GitHub, gh-grep, Playwright는 인터넷 의존이므로 각 셀에서 자동으로 연결 체크 후 스킵됩니다.

## Next Steps

→ `3_connect_ide_clients.ipynb` — Configure your IDE to use these MCP server Routes (direct access)
→ `../2_ai_gateway/2_enable_maas.ipynb` — Register MCP servers with MaaS gateway for unified access with auth